In [1]:
import pygame
import random
import copy

<frozen importlib._bootstrap>:491: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.


pygame 2.6.1 (SDL 2.32.74, Python 3.14.8)
Hello from the pygame community. https://www.pygame.org/contribute.html


The Four Core Rules

At each tick (generation), the rules apply simultaneously to every cell on the grid based on its current neighbors:
1. Underpopulation: A live cell with fewer than two live neighbors dies.
2. Survival: A live cell with two or three live neighbors lives on to the next generation.
3. Overpopulation: A live cell with more than three live neighbors dies.
4. Reproduction: A dead cell with exactly three live neighbors becomes a live cell.

In [20]:
class Board():
    def __init__(self, size=(1000,1000), F=0.2):
        self.size = size
        self.grid = [[0 for j in range(size[1])] for i in range(size[0])]
        self.F = F
        
    def initialize_state(self, live_cells=None):
        "[(xi,yi)]"
        if live_cells is not None:
            for cell in list(live_cells):
                self.grid[cell[0]][cell[1]] = 1
        else: # randomly init a fraction F of total cells in grid
            total_cells = self.size[0] * self.size[1]
            initial_alive_count = total_cells * self.F
            initial_alive_count = int(initial_alive_count)
            for _ in range(0, initial_alive_count):
                x = random.randint(0,self.size[0]-1)
                y = random.randint(0, self.size[1]-1)
                self.grid[x][y] = 1

    def step(self):
        grid_copy = copy.deepcopy(self.grid)
        # update grid based on 8 neighboring cells and 4 core rules:
        def get_live_neighbors(x,y):
            x_shift = [-1, 0, 1]
            y_shift = [-1, 0, 1]
            alive = 0
            for delta_x in x_shift:
                for delta_y in y_shift:
                    if not (delta_x==0 and delta_y==0):
                        if 0<=x+delta_x<self.size[0] and 0<=y+delta_y<self.size[1]:
                            if grid_copy[x+delta_x][y+delta_y] == 1:
                                alive += 1
            return alive
            

        for x in range(0, self.size[0]):
            for y in range(0, self.size[1]):
                cell_state = grid_copy[x][y]
                alive_neighbor_count = get_live_neighbors(x,y)
                if cell_state == 1:
                    # underpopulation:
                    if alive_neighbor_count < 2:
                        self.grid[x][y] = 0

                    # overpopulation
                    if alive_neighbor_count >3:
                        self.grid[x][y] = 0

                    # reproduction
                elif cell_state == 0:
                    if alive_neighbor_count == 3:
                        self.grid[x][y] = 1

In [58]:
WINDOW_X = 1000 # goes across window left to right. this many PIXELS
WINDOW_Y = 1000 # goes up to down on window. this many PIXELS

# Smaller grid = larger, easily visible cells
GRID_ROWS = 50   # number of ROWS in grid
GRID_COLS = 50  # number of columns in grid

CELL_SIZE_X = WINDOW_X // GRID_COLS  # number of pixels PER CELL in WIN_X direction
CELL_SIZE_Y = WINDOW_Y // GRID_ROWS  # number of pixels PER CELL in WIN_Y direction

"""
logical grid cell (x,y) -> xth row, yth col IN GRID coords
CELL_SIZE_Y * x -> starting top left corner coord of rectangle pixel in WIN_Y direction
CELL_SIZE_X * y -> starting top left corner coord of rectangle pixel in WIN_X direction 
"""

# Start the glider near the top-left so you can watch it crawl across
r, c = 5, 5  

glider_matrix_cells = [
    (r,     c + 1), 
    (r + 1, c + 2), 
    (r + 2, c    ), (r + 2, c + 1), (r + 2, c + 2)
]

r, c = 20, 10  # Starting row and column

lwss_matrix_cells = [
    (r,     c + 1), (r,     c + 4),
    (r + 1, c    ),
    (r + 2, c    ), (r + 2, c + 4),
    (r + 3, c + 1), (r + 3, c + 2), (r + 3, c + 3), (r + 3, c + 4)
]


r, c = 25, 25  # Center of the 50x50 grid

pulsar_offsets = [
    (-2, -4), (-1, -4), (1, -4), (2, -4),
    (-4, -2), (-3, -2), (3, -2), (4, -2),
    (-4, -1), (-3, -1), (3, -1), (4, -1),
    (-2, 1), (-1, 1), (1, 1), (2, 1),
    (-4, 2), (-3, 2), (3, 2), (4, 2),
    (-4, 3), (-3, 3), (3, 3), (4, 3),
    (-2, 4), (-1, 4), (1, 4), (2, 4),
    (-2, -6), (-1, -6), (1, -6), (2, -6),
    (-6, -2), (-6, -1), (-6, 1), (-6, 2),
    (6, -2), (6, -1), (6, 1), (6, 2),
    (-2, 6), (-1, 6), (1, 6), (2, 6)
]

pulsar_matrix_cells = [(r + dr, c + dc) for dr, dc in pulsar_offsets]





r, c = 25, 20  # Positioned horizontally across the middle

pentadecathlon_offsets = [
    (0, 0), (0, 1), (0, 2), (0, 3), (0, 4), 
    (0, 5), (0, 6), (0, 7), (0, 8), (0, 9),
    # with specific gaps built into the line
]

# Clean explicit matrix cells for the Pentadecathlon
pentadecathlon_matrix_cells = [
    (r, c), (r, c+1), (r, c+2), 
    (r, c+4), (r, c+5), (r, c+6), (r, c+7), 
    (r, c+9), (r, c+10), (r, c+11),
    # flanks
    (r-1, c+2), (r-1, c+9),
    (r+1, c+2), (r+1, c+9)
]


In [70]:
pygame.init()
board = Board((GRID_ROWS, GRID_COLS), F=0.4)
board.initialize_state()

running = True
screen = pygame.display.set_mode((WINDOW_X,WINDOW_Y))
pygame.display.set_caption("Conway's Game of Life")
clock = pygame.time.Clock()

# board's (x,y) equivalent position on window is window's (y,x)


def draw_grid():
    for x in range(GRID_ROWS):
        for y in range(GRID_COLS):
            if board.grid[x][y]:
                pygame.draw.rect(screen, (255,255,255), (CELL_SIZE_X * y,CELL_SIZE_Y * x,CELL_SIZE_X,CELL_SIZE_Y))

while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
    
    # reset frame
    screen.fill((0, 0, 0))
    draw_grid()
    board.step()
    pygame.display.flip()
    clock.tick(6)
pygame.quit()